In [0]:
order_number        (FK)
customer_id         (FK)
product_id / item_id
order_date_key      (FK)

qty
unit_price (if available)
discount
promo_quantity

total_amount (derived)

In [0]:
#read tables

sales_orders_df = spark.read.table("retaildataplatform.silver.sales_orders")
order_products_df = spark.read.table("retaildataplatform.silver.order_products")
promotions_df = spark.read.table("retaildataplatform.silver.promotions")




In [0]:
%sql
select * from retaildataplatform.silver.sales_orders
--where order_number = 317569280



--20190812

In [0]:
%sql
select * from retaildataplatform.silver.order_products
where id = 'AVpge6k2LJeJML43OhAl' and  order_number = 317569280

In [0]:
%sql
select * from 
(
select * ,
     row_number() over (
        partition by order_number
        order by order_timestamp desc 
     ) as rank
     from 


retaildataplatform.silver.sales_orders

) t

--where t.rank = 1

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.function as F 

win = Window.partitionBy("order_number").orderBy("order_timestamp").desc()


df = df.withColumn("rank", F.row_number().over(win) )


In [0]:
%sql
with joined 
(
    
     select 
     s.order_number, 
     s.customer_id, 
     p.id as product_id,
     s.order_timestamp as order_date_key,
     p.price,
     p.qty,

     row_number() over (
        partition by s.order_number
        order by s.order_timestamp desc 
     ) as rank

     from retaildataplatform.silver.sales_orders as s 
     left join retaildataplatform.silver.order_products p on s.order_number=p.order_number 
) 
     
     
select j.order_number,  j.customer_id, j.product_id, j.order_date_key,j.price, j.qty, pm.discount, pm.promo_quantity

 from joined as j 
     
     join retaildataplatform.silver.promotions pm 
     
     on j.order_number=pm.order_number and j.product_id=pm.promo_product_id
     where j.rank = 1 




In [0]:
joined_df = (sales_orders_df.alias("so")
    .join(order_products_df.alias("op"),
        F.col("so.order_number") == F.col("op.order_number"),
        "left"
    )
    .join(promotions_df.alias("p"),F.col("so.order_number") == F.col("p.order_number"),
        "left"
    )
)

In [0]:
%sql
select order_number, count(*) from retaildataplatform.silver.promotions
group by order_number
having count(*) > 1



-- 317568456


In [0]:
retaildataplatform.silver.sales
retaildataplatform.silver.order_products



In [0]:
sales_orders_df.display() # order_number, customer_id, order_timestamp (create order key out of it)

In [0]:
order_products_df.display() # id as product_id, qty, price as unit_price, 

In [0]:
promotions_df.display() # promo_quantity, discount

In [0]:
fact_sales
retaildataplatform.silver.sales_orders -> order_number
customer_id



